# Exercise 3 — Edit, save, and extract

**Goal:** make a small information change safely, write a **new** IFC file, and verify it by reopening. Allow 20–25 minutes. We change a teaching label, not a material or engineering property.

Every run starts from the original sample. New files appear in `exports/` in Binder; download them before the temporary session ends.

In [ ]:
from pathlib import Path
from uuid import uuid4
from compas_ifc.bim import BuildingInformationModel

IFC_PATH = Path("sample_building.ifc")  # CHANGE for another public teaching model
if not IFC_PATH.is_file():
    raise FileNotFoundError(f"Missing {IFC_PATH}")
model = BuildingInformationModel(str(IFC_PATH), load_geometries=False, rectify_placements=False)
walls = model.get_elements_by_type("IfcWall")
if not walls:
    raise ValueError("Start with the bundled sample or choose a model containing a wall.")
wall = walls[0]
original_id, original_name = wall.global_id, wall.name
print(original_id, original_name)

## 1. Change one Name

The Python object updates the model in memory. The source IFC on disk is still untouched until we save a new path. Predict whether `GlobalId` will change.

In [ ]:
NEW_NAME = "My course wall"  # CHANGE this teaching label
wall.name = NEW_NAME
print("Old Name:", original_name)
print("New Name:", wall.name)
print("Same GlobalId:", wall.global_id == original_id)

## 2. Save a new file and reopen it

The short random suffix keeps repeated runs from replacing previous work. Reopening verifies the saved file rather than merely inspecting the in-memory object.

In [ ]:
output_dir = Path("exports")
output_dir.mkdir(exist_ok=True)
run_id = uuid4().hex[:8]
edited_path = output_dir / f"renamed_{run_id}.ifc"
if edited_path.resolve() == IFC_PATH.resolve() or edited_path.exists():
    raise FileExistsError("Choose a new output path before saving.")
model.save(str(edited_path))

reopened = BuildingInformationModel(str(edited_path), load_geometries=False, rectify_placements=False)
checked = reopened.get_element_by_global_id(original_id)
assert checked is not None and checked.name == NEW_NAME
print("Saved and checked:", edited_path)
print("GlobalId:", checked.global_id, "Name:", checked.name)

## 3. Extract a smaller IFC

`extract` makes a standalone IFC containing the selected wall and required spatial context. It is a **new model**. We use the reopened edited file so the new label should be included. The sample is small; this matters more on a larger model.

In [ ]:
subset_path = output_dir / f"wall_subset_{run_id}.ifc"
if subset_path.exists() or subset_path.resolve() == IFC_PATH.resolve():
    raise FileExistsError("Choose a new subset path.")
subset = reopened.extract(reopened.get_elements_by_type("IfcWall"), path=str(subset_path))
subset_check = BuildingInformationModel(str(subset_path), load_geometries=False, rectify_placements=False)
print("Subset:", subset_path)
print("Walls:", len(subset_check.get_elements_by_type("IfcWall")), "Storeys:", len(subset_check.storeys))
print("Renamed object in subset:", subset_check.get_element_by_global_id(original_id).name)

## Your turn

1. Change `NEW_NAME`, rerun from the first cell, and verify the new file. Explain why you must rerun from the first cell.
2. Find your two output files in Binder's file browser and download them.
3. Compare source, edited, and subset files: which share the wall's GlobalId? Which contain your new Name?

**Check:** saving a file is not the same as validating an engineering model. This exercise verifies one identity and one label. [Upstream granular export tutorial](https://compas.dev/compas_ifc/latest/tutorials/05_granular_export.html).